<a href="https://colab.research.google.com/github/RayR2333/hong-kong-weather-etl/blob/main/SPY%E7%9A%84%E8%8D%89%E7%A8%BF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd

# 下载 SPY 全部历史数据（auto_adjust=True 会使用复权价格）
spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 计算每一天的“前20日收益”（简单收益率）
spy['ret_20d'] = spy['Close'].pct_change(20)

# 查看结果
print(spy[['Close', 'ret_20d']].tail(10))
print(f"有效数据从 {spy['ret_20d'].first_valid_index().date()} 开始")

# 可选：保存到 CSV
spy[['Close', 'ret_20d']].to_csv('spy_20d_returns.csv')

[*********************100%***********************]  1 of 1 completed


Price            Close   ret_20d
Ticker             SPY          
Date                            
2026-11-02  752.099976 -0.029336
2026-11-03  754.349976 -0.031755
2026-11-04  766.809998 -0.013394
2026-11-05  733.000000 -0.052886
2026-11-06  735.289978 -0.055589
2026-11-09  752.690002 -0.032656
2026-11-10  749.919983 -0.036105
2026-11-11  759.809998 -0.012721
2026-11-12  757.119995 -0.019008
2026-11-13  754.369995 -0.011634
有效数据从 1993-03-01 开始


In [ ]:
import yfinance as yf
import pandas as pd

# 1. 下载数据并计算20日收益
spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)
spy['ret_20d'] = spy['Close'].pct_change(20)

# 2. 去掉前20天的 NaN
returns = spy['ret_20d'].dropna()

# 3. 基本统计
print("=== 基本统计 ===")
print(f"平均数 (Mean):   {returns.mean():.4%}")
print(f"中位数 (Median): {returns.median():.4%}")
print(f"标准差 (Std):    {returns.std():.4%}")
print(f"样本数量:        {len(returns)}")

# 4. 前100大（最高收益）
print("\n=== 前100大（最高20日收益） ===")
top_100 = returns.nlargest(100)
print(top_100.head(10))          # 先看前10个
print(f"... 共 {len(top_100)} 个")

# 5. 后100小（最低收益）
print("\n=== 后100小（最低20日收益） ===")
bottom_100 = returns.nsmallest(100)
print(bottom_100.head(10))
print(f"... 共 {len(bottom_100)} 个")

# 6. 如果想同时看到日期和收盘价
print("\n=== 前100大（带日期） ===")
top_100_df = spy.loc[top_100.index, ['Close', 'ret_20d']].sort_values('ret_20d', ascending=False)
print(top_100_df.head(10))

print("\n=== 后100小（带日期） ===")
bottom_100_df = spy.loc[bottom_100.index, ['Close', 'ret_20d']].sort_values('ret_20d', ascending=True)
print(bottom_100_df.head(10))

[*********************100%***********************]  1 of 1 completed

=== 基本统计 ===
平均数 (Mean):   0.9147%
中位数 (Median): 1.4080%
标准差 (Std):    4.4132%
样本数量:        8487

=== 前100大（最高20日收益） ===
Date
2009-04-06    0.236212
2009-04-03    0.231328
2020-04-20    0.230725
2020-04-21    0.224669
2009-04-02    0.221325
2020-04-17    0.198809
2002-11-06    0.191293
2020-04-14    0.190155
2002-08-20    0.180612
2020-04-30    0.180093
Name: ret_20d, dtype: float64
... 共 100 个

=== 后100小（最低20日收益） ===
Date
2020-03-20   -0.309867
2020-03-23   -0.304445
2008-10-10   -0.294057
2020-03-18   -0.290654
2020-03-16   -0.289544
2020-03-19   -0.286215
2008-10-24   -0.279768
2008-10-09   -0.273166
2020-03-12   -0.264685
2020-03-17   -0.249250
Name: ret_20d, dtype: float64
... 共 100 个

=== 前100大（带日期） ===
Price            Close   ret_20d
Ticker             SPY          
Date                            
2009-04-06   61.263580  0.236212
2009-04-03   61.747246  0.231328
2020-04-20  257.544495  0.230725
2020-04-21  249.724625  0.224669
2009-04-02   61.138973  0.221325
2020-04-17  262.1

In [ ]:
import numpy as np
import pandas as pd

# 假设 returns 已经是去掉 NaN 的 ret_20d 序列
returns = spy['ret_20d'].dropna()

# 生成 101 个等间距边界 → 得到 100 个区间
bins = np.linspace(returns.min(), returns.max(), 101)

# 统计每个区间的样本数量
counts, bin_edges = np.histogram(returns, bins=bins)

# 整理成可读的 DataFrame
bin_df = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})

# 计算占比
bin_df['pct'] = bin_df['count'] / bin_df['count'].sum()

# 显示全部结果（或只看有数据的区间）
print(bin_df.to_string(index=False))

# 可选：只看有样本的区间
print("\n=== 仅显示有样本的区间 ===")
print(bin_df[bin_df['count'] > 0].to_string(index=False))

 bin_left  bin_right  count      pct
-0.309867  -0.304407      2 0.000236
-0.304407  -0.298946      0 0.000000
-0.298946  -0.293485      1 0.000118
-0.293485  -0.288024      2 0.000236
-0.288024  -0.282563      1 0.000118
-0.282563  -0.277103      1 0.000118
-0.277103  -0.271642      1 0.000118
-0.271642  -0.266181      0 0.000000
-0.266181  -0.260720      1 0.000118
-0.260720  -0.255260      0 0.000000
-0.255260  -0.249799      0 0.000000
-0.249799  -0.244338      3 0.000353
-0.244338  -0.238877      1 0.000118
-0.238877  -0.233416      1 0.000118
-0.233416  -0.227956      0 0.000000
-0.227956  -0.222495      1 0.000118
-0.222495  -0.217034      1 0.000118
-0.217034  -0.211573      2 0.000236
-0.211573  -0.206112      2 0.000236
-0.206112  -0.200652      2 0.000236
-0.200652  -0.195191      2 0.000236
-0.195191  -0.189730      2 0.000236
-0.189730  -0.184269      2 0.000236
-0.184269  -0.178808      2 0.000236
-0.178808  -0.173348      4 0.000471
-0.173348  -0.167887      4 0.000471
-

In [ ]:
import yfinance as yf
import pandas as pd

# 下载 SPY 全部历史（复权价格）
spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 计算每一天的前60日收益
spy['ret_60d'] = spy['Close'].pct_change(60)

# 去掉前60天的 NaN
returns_60 = spy['ret_60d'].dropna()

print(f"有效样本数量: {len(returns_60)}")
print(f"起始日期: {returns_60.index[0].date()}")
print(f"结束日期: {returns_60.index[-1].date()}")

# 基本统计
print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_60.mean():.4%}")
print(f"中位数 (Median): {returns_60.median():.4%}")
print(f"标准差 (Std):    {returns_60.std():.4%}")
print(f"最小值:          {returns_60.min():.4%}")
print(f"最大值:          {returns_60.max():.4%}")

# 前100大 / 后100小
print("\n=== 前100大 ===")
print(returns_60.nlargest(10))

print("\n=== 后100小 ===")
print(returns_60.nsmallest(10))

# 可选：保存结果
spy[['Close', 'ret_60d']].to_csv('spy_60d_returns.csv')

[*********************100%***********************]  1 of 1 completed

有效样本数量: 8447
起始日期: 1993-04-27
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   2.7428%
中位数 (Median): 3.5625%
标准差 (Std):    7.2573%
最小值:          -41.0039%
最大值:          39.7892%

=== 前100大 ===
Date
2020-06-17    0.397892
2009-06-01    0.387330
2009-06-02    0.386083
2009-06-03    0.384822
2020-06-16    0.367832
2020-06-10    0.337819
2009-06-04    0.319199
2009-06-05    0.310940
2020-06-08    0.310308
2009-05-28    0.306847
Name: ret_60d, dtype: float64

=== 后100小 ===
Date
2008-11-20   -0.410039
2008-11-21   -0.385665
2008-11-19   -0.356529
2008-11-17   -0.336948
2008-12-01   -0.336238
2008-11-24   -0.335957
2008-10-27   -0.330723
2008-11-25   -0.326855
2008-12-02   -0.324643
2008-11-12   -0.320287
Name: ret_60d, dtype: float64


In [ ]:
import numpy as np
import pandas as pd

# 假设 returns_60 已经是去掉 NaN 的 ret_60d 序列
# returns_60 = spy['ret_60d'].dropna()

# 生成 101 个等间距边界 → 得到 100 个区间
bins = np.linspace(returns_60.min(), returns_60.max(), 101)

# 统计每个区间的样本数量
counts, bin_edges = np.histogram(returns_60, bins=bins)

# 整理成 DataFrame
bin_df_60 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_60['pct'] = bin_df_60['count'] / bin_df_60['count'].sum()

print(f"总样本数: {bin_df_60['count'].sum()}")
print(bin_df_60.to_string(index=False))

# 只显示有样本的区间
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_60[bin_df_60['count'] > 0].to_string(index=False))

总样本数: 8447
 bin_left  bin_right  count      pct
-0.410039  -0.401959      1 0.000118
-0.401959  -0.393880      0 0.000000
-0.393880  -0.385801      0 0.000000
-0.385801  -0.377721      1 0.000118
-0.377721  -0.369642      0 0.000000
-0.369642  -0.361563      0 0.000000
-0.361563  -0.353484      1 0.000118
-0.353484  -0.345404      0 0.000000
-0.345404  -0.337325      0 0.000000
-0.337325  -0.329246      4 0.000474
-0.329246  -0.321166      2 0.000237
-0.321166  -0.313087      2 0.000237
-0.313087  -0.305008      3 0.000355
-0.305008  -0.296928      2 0.000237
-0.296928  -0.288849      6 0.000710
-0.288849  -0.280770      5 0.000592
-0.280770  -0.272690      3 0.000355
-0.272690  -0.264611      3 0.000355
-0.264611  -0.256532      4 0.000474
-0.256532  -0.248453      8 0.000947
-0.248453  -0.240373     10 0.001184
-0.240373  -0.232294      5 0.000592
-0.232294  -0.224215      3 0.000355
-0.224215  -0.216135      6 0.000710
-0.216135  -0.208056      5 0.000592
-0.208056  -0.199977      7

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# 下载数据
spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 计算前252日收益
spy['ret_252d'] = spy['Close'].pct_change(252)

# 去掉 NaN
returns_252 = spy['ret_252d'].dropna()

print(f"有效样本数量: {len(returns_252)}")
print(f"起始日期: {returns_252.index[0].date()}")
print(f"结束日期: {returns_252.index[-1].date()}")

# 基本统计
print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_252.mean():.4%}")
print(f"中位数 (Median): {returns_252.median():.4%}")
print(f"标准差 (Std):    {returns_252.std():.4%}")
print(f"最小值:          {returns_252.min():.4%}")
print(f"最大值:          {returns_252.max():.4%}")

# 前100大 / 后100小
print("\n=== 前100大（最高252日收益） ===")
print(returns_252.nlargest(10))

print("\n=== 后100小（最低252日收益） ===")
print(returns_252.nsmallest(10))

[*********************100%***********************]  1 of 1 completed

有效样本数量: 8255
起始日期: 1994-01-27
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   12.1408%
中位数 (Median): 14.4386%
标准差 (Std):    16.6871%
最小值:          -47.3541%
最大值:          77.5045%

=== 前100大（最高252日收益） ===
Date
2021-03-23    0.775045
2021-03-22    0.743382
2010-03-09    0.719421
2010-03-05    0.699053
2010-03-08    0.696393
2021-03-16    0.681482
2021-04-06    0.662568
2021-03-18    0.661629
2021-03-19    0.655037
2021-04-01    0.653603
Name: ret_252d, dtype: float64

=== 后100小（最低252日收益） ===
Date
2009-03-05   -0.473541
2009-03-03   -0.462498
2009-03-09   -0.462267
2009-03-06   -0.461477
2009-03-02   -0.459727
2008-11-20   -0.456164
2009-03-04   -0.447654
2009-02-27   -0.446851
2008-10-27   -0.441928
2009-02-26   -0.439733
Name: ret_252d, dtype: float64


In [ ]:
# 分箱
bins = np.linspace(returns_252.min(), returns_252.max(), 101)
counts, bin_edges = np.histogram(returns_252, bins=bins)

bin_df_252 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_252['pct'] = bin_df_252['count'] / bin_df_252['count'].sum()

print(f"\n总样本数: {bin_df_252['count'].sum()}")
print(f"区间宽度: {bins[1] - bins[0]:.6f}")
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_252[bin_df_252['count'] > 0].to_string(index=False))


总样本数: 8255
区间宽度: 0.012486

=== 仅显示有样本的区间 ===
 bin_left  bin_right  count      pct
-0.473541  -0.461055      4 0.000485
-0.461055  -0.448569      2 0.000242
-0.448569  -0.436084      7 0.000848
-0.436084  -0.423598      3 0.000363
-0.423598  -0.411112      7 0.000848
-0.411112  -0.398626     17 0.002059
-0.398626  -0.386140     15 0.001817
-0.386140  -0.373654     23 0.002786
-0.373654  -0.361168     28 0.003392
-0.361168  -0.348683     17 0.002059
-0.348683  -0.336197     23 0.002786
-0.336197  -0.323711     14 0.001696
-0.323711  -0.311225      6 0.000727
-0.311225  -0.298739     13 0.001575
-0.298739  -0.286253     18 0.002180
-0.286253  -0.273767      9 0.001090
-0.273767  -0.261282     19 0.002302
-0.261282  -0.248796     18 0.002180
-0.248796  -0.236310     28 0.003392
-0.236310  -0.223824     37 0.004482
-0.223824  -0.211338     39 0.004724
-0.211338  -0.198852     64 0.007753
-0.198852  -0.186366     68 0.008237
-0.186366  -0.173881     71 0.008601
-0.173881  -0.161395     68 0

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 约3个自然年
spy['ret_756d'] = spy['Close'].pct_change(756)

returns_756 = spy['ret_756d'].dropna()

print(f"有效样本数量: {len(returns_756)}")
print(f"起始日期: {returns_756.index[0].date()}")
print(f"结束日期: {returns_756.index[-1].date()}")

print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_756.mean():.4%}")
print(f"中位数 (Median): {returns_756.median():.4%}")
print(f"标准差 (Std):    {returns_756.std():.4%}")
print(f"最小值:          {returns_756.min():.4%}")
print(f"最大值:          {returns_756.max():.4%}")

print("\n=== 前100大 ===")
print(returns_756.nlargest(10))

print("\n=== 后100小 ===")
print(returns_756.nsmallest(10))

[*********************100%***********************]  1 of 1 completed

有效样本数量: 7751
起始日期: 1996-01-26
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   40.5068%
中位数 (Median): 41.3088%
标准差 (Std):    37.4979%
最小值:          -43.4632%
最大值:          134.5833%

=== 前100大 ===
Date
1999-07-15    1.345833
1998-03-20    1.345174
1998-04-03    1.344752
1998-04-17    1.343292
1998-04-02    1.341688
1998-04-15    1.341476
1999-07-14    1.336832
1997-12-05    1.336346
1998-04-20    1.331134
1999-07-16    1.330736
Name: ret_756d, dtype: float64

=== 后100小 ===
Date
2009-03-09   -0.434632
2009-03-05   -0.432408
2009-03-06   -0.428801
2009-03-03   -0.424657
2003-03-31   -0.422020
2003-04-01   -0.416617
2009-03-02   -0.415151
2002-07-23   -0.412385
2009-03-04   -0.410981
2003-03-11   -0.406810
Name: ret_756d, dtype: float64


In [ ]:
bins = np.linspace(returns_756.min(), returns_756.max(), 101)
counts, bin_edges = np.histogram(returns_756, bins=bins)

bin_df_756 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_756['pct'] = bin_df_756['count'] / bin_df_756['count'].sum()

print(f"\n总样本数: {bin_df_756['count'].sum()}")
print(f"区间宽度: {(returns_756.max() - returns_756.min()) / 100:.6f}")
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_756[bin_df_756['count'] > 0].to_string(index=False))


总样本数: 7751
区间宽度: 0.017805

=== 仅显示有样本的区间 ===
 bin_left  bin_right  count      pct
-0.434632  -0.416828      5 0.000645
-0.416828  -0.399023     10 0.001290
-0.399023  -0.381218     25 0.003225
-0.381218  -0.363414     32 0.004128
-0.363414  -0.345609     39 0.005032
-0.345609  -0.327804     54 0.006967
-0.327804  -0.310000     55 0.007096
-0.310000  -0.292195     91 0.011740
-0.292195  -0.274390     62 0.007999
-0.274390  -0.256586     60 0.007741
-0.256586  -0.238781     50 0.006451
-0.238781  -0.220976     52 0.006709
-0.220976  -0.203172     58 0.007483
-0.203172  -0.185367     81 0.010450
-0.185367  -0.167562     79 0.010192
-0.167562  -0.149758    106 0.013676
-0.149758  -0.131953     62 0.007999
-0.131953  -0.114148     58 0.007483
-0.114148  -0.096344     49 0.006322
-0.096344  -0.078539     58 0.007483
-0.078539  -0.060735     35 0.004516
-0.060735  -0.042930     39 0.005032
-0.042930  -0.025125     49 0.006322
-0.025125  -0.007321     39 0.005032
-0.007321   0.010484     47 0

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 约2.5个自然年
spy['ret_630d'] = spy['Close'].pct_change(630)

returns_630 = spy['ret_630d'].dropna()

print(f"有效样本数量: {len(returns_630)}")
print(f"起始日期: {returns_630.index[0].date()}")
print(f"结束日期: {returns_630.index[-1].date()}")

print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_630.mean():.4%}")
print(f"中位数 (Median): {returns_630.median():.4%}")
print(f"标准差 (Std):    {returns_630.std():.4%}")
print(f"最小值:          {returns_630.min():.4%}")
print(f"最大值:          {returns_630.max():.4%}")

print("\n=== 前100大 ===")
print(returns_630.nlargest(10))

print("\n=== 后100小 ===")
print(returns_630.nsmallest(10))

[*********************100%***********************]  1 of 1 completed

有效样本数量: 7877
起始日期: 1995-07-28
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   33.1613%
中位数 (Median): 34.7504%
标准差 (Std):    31.8847%
最小值:          -46.3141%
最大值:          113.0881%

=== 前100大 ===
Date
1997-07-30    1.130881
1997-07-31    1.116146
1997-07-24    1.108566
1997-07-22    1.108234
1997-07-28    1.106466
1997-07-29    1.105056
1997-07-03    1.102249
1997-07-23    1.100862
1997-07-17    1.099447
1997-07-25    1.098780
Name: ret_630d, dtype: float64

=== 后100小 ===
Date
2002-10-09   -0.463141
2002-10-07   -0.456965
2009-03-09   -0.455104
2003-03-11   -0.453174
2002-10-08   -0.451646
2002-09-30   -0.450361
2003-03-12   -0.450071
2009-03-06   -0.447575
2009-03-05   -0.445244
2002-10-10   -0.442267
Name: ret_630d, dtype: float64


In [ ]:
bins = np.linspace(returns_630.min(), returns_630.max(), 101)
counts, bin_edges = np.histogram(returns_630, bins=bins)

bin_df_630 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_630['pct'] = bin_df_630['count'] / bin_df_630['count'].sum()

print(f"\n总样本数: {bin_df_630['count'].sum()}")
print(f"区间宽度: {(returns_630.max() - returns_630.min()) / 100:.6f}")
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_630[bin_df_630['count'] > 0].to_string(index=False))


总样本数: 7877
区间宽度: 0.015940

=== 仅显示有样本的区间 ===
 bin_left  bin_right  count      pct
-0.463141  -0.447200      8 0.001016
-0.447200  -0.431260     12 0.001523
-0.431260  -0.415320     21 0.002666
-0.415320  -0.399380     17 0.002158
-0.399380  -0.383440     26 0.003301
-0.383440  -0.367499     33 0.004189
-0.367499  -0.351559     42 0.005332
-0.351559  -0.335619     50 0.006348
-0.335619  -0.319679     45 0.005713
-0.319679  -0.303739     64 0.008125
-0.303739  -0.287798     43 0.005459
-0.287798  -0.271858     33 0.004189
-0.271858  -0.255918     51 0.006475
-0.255918  -0.239978     47 0.005967
-0.239978  -0.224037     88 0.011172
-0.224037  -0.208097     79 0.010029
-0.208097  -0.192157     49 0.006221
-0.192157  -0.176217     60 0.007617
-0.176217  -0.160277     61 0.007744
-0.160277  -0.144336     46 0.005840
-0.144336  -0.128396     52 0.006601
-0.128396  -0.112456     58 0.007363
-0.112456  -0.096516     63 0.007998
-0.096516  -0.080576     46 0.005840
-0.080576  -0.064635     28 0

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 约2个自然年
spy['ret_504d'] = spy['Close'].pct_change(504)

returns_504 = spy['ret_504d'].dropna()

print(f"有效样本数量: {len(returns_504)}")
print(f"起始日期: {returns_504.index[0].date()}")
print(f"结束日期: {returns_504.index[-1].date()}")

print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_504.mean():.4%}")
print(f"中位数 (Median): {returns_504.median():.4%}")
print(f"标准差 (Std):    {returns_504.std():.4%}")
print(f"最小值:          {returns_504.min():.4%}")
print(f"最大值:          {returns_504.max():.4%}")

print("\n=== 前100大 ===")
print(returns_504.nlargest(10))

print("\n=== 后100小 ===")
print(returns_504.nsmallest(10))

[*********************100%***********************]  1 of 1 completed

有效样本数量: 8003
起始日期: 1995-01-27
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   25.9247%
中位数 (Median): 27.2273%
标准差 (Std):    26.9996%
最小值:          -49.5264%
最大值:          107.5851%

=== 前100大 ===
Date
2022-03-22    1.075851
2011-03-08    1.030717
2011-03-04    1.008683
2022-03-21    0.999380
2011-03-07    0.989443
2011-03-02    0.953517
2011-03-03    0.941176
1998-07-14    0.940323
2011-03-01    0.934713
1998-07-15    0.931902
Name: ret_504d, dtype: float64

=== 后100小 ===
Date
2009-03-09   -0.495264
2009-03-05   -0.486356
2009-03-06   -0.484943
2009-03-02   -0.475955
2009-03-03   -0.472988
2009-02-23   -0.466254
2009-03-10   -0.465329
2009-03-11   -0.462649
2002-07-23   -0.457325
2009-03-04   -0.455318
Name: ret_504d, dtype: float64


In [ ]:
bins = np.linspace(returns_504.min(), returns_504.max(), 101)
counts, bin_edges = np.histogram(returns_504, bins=bins)

bin_df_504 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_504['pct'] = bin_df_504['count'] / bin_df_504['count'].sum()

print(f"\n总样本数: {bin_df_504['count'].sum()}")
print(f"区间宽度: {(returns_504.max() - returns_504.min()) / 100:.6f}")
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_504[bin_df_504['count'] > 0].to_string(index=False))


总样本数: 8003
区间宽度: 0.015711

=== 仅显示有样本的区间 ===
 bin_left  bin_right  count      pct
-0.495264  -0.479553      3 0.000375
-0.479553  -0.463842      4 0.000500
-0.463842  -0.448131      4 0.000500
-0.448131  -0.432420     13 0.001624
-0.432420  -0.416709     14 0.001749
-0.416709  -0.400997     22 0.002749
-0.400997  -0.385286     47 0.005873
-0.385286  -0.369575     50 0.006248
-0.369575  -0.353864     61 0.007622
-0.353864  -0.338153     40 0.004998
-0.338153  -0.322442     40 0.004998
-0.322442  -0.306730     31 0.003874
-0.306730  -0.291019     42 0.005248
-0.291019  -0.275308     34 0.004248
-0.275308  -0.259597     43 0.005373
-0.259597  -0.243886     30 0.003749
-0.243886  -0.228175     50 0.006248
-0.228175  -0.212464     49 0.006123
-0.212464  -0.196752     53 0.006623
-0.196752  -0.181041     43 0.005373
-0.181041  -0.165330     66 0.008247
-0.165330  -0.149619     65 0.008122
-0.149619  -0.133908     63 0.007872
-0.133908  -0.118197     54 0.006747
-0.118197  -0.102485     38 0

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download('SPY', start='1993-01-01', auto_adjust=True)

# 约5个自然年
spy['ret_1260d'] = spy['Close'].pct_change(1260)

returns_1260 = spy['ret_1260d'].dropna()

print(f"有效样本数量: {len(returns_1260)}")
print(f"起始日期: {returns_1260.index[0].date()}")
print(f"结束日期: {returns_1260.index[-1].date()}")

print("\n=== 基本统计 ===")
print(f"平均数 (Mean):   {returns_1260.mean():.4%}")
print(f"中位数 (Median): {returns_1260.median():.4%}")
print(f"标准差 (Std):    {returns_1260.std():.4%}")
print(f"最小值:          {returns_1260.min():.4%}")
print(f"最大值:          {returns_1260.max():.4%}")

print("\n=== 前100大 ===")
print(returns_1260.nlargest(10))

print("\n=== 后100小 ===")
print(returns_1260.nsmallest(10))

[*********************100%***********************]  1 of 1 completed

有效样本数量: 7247
起始日期: 1998-01-26
结束日期: 2026-11-13

=== 基本统计 ===
平均数 (Mean):   70.6922%
中位数 (Median): 72.0933%
标准差 (Std):    60.5425%
最小值:          -35.5249%
最大值:          248.0682%

=== 前100大 ===
Date
1999-12-28    2.480682
1999-12-29    2.478859
1999-12-06    2.470665
1999-12-31    2.463790
1999-12-03    2.462731
1999-12-30    2.458263
1999-11-18    2.457236
1999-12-23    2.448721
1999-12-27    2.441602
1999-07-16    2.435612
Name: ret_1260d, dtype: float64

=== 后100小 ===
Date
2009-03-09   -0.355249
2009-03-06   -0.345388
2009-03-05   -0.344833
2009-03-03   -0.335439
2009-03-02   -0.323776
2009-03-04   -0.315689
2009-03-10   -0.308377
2009-03-11   -0.301077
2009-02-27   -0.291388
2009-02-23   -0.284113
Name: ret_1260d, dtype: float64


In [ ]:
bins = np.linspace(returns_1260.min(), returns_1260.max(), 101)
counts, bin_edges = np.histogram(returns_1260, bins=bins)

bin_df_1260 = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': counts
})
bin_df_1260['pct'] = bin_df_1260['count'] / bin_df_1260['count'].sum()

print(f"\n总样本数: {bin_df_1260['count'].sum()}")
print(f"区间宽度: {(returns_1260.max() - returns_1260.min()) / 100:.6f}")
print("\n=== 仅显示有样本的区间 ===")
print(bin_df_1260[bin_df_1260['count'] > 0].to_string(index=False))


总样本数: 7247
区间宽度: 0.028359

=== 仅显示有样本的区间 ===
 bin_left  bin_right  count      pct
-0.355249  -0.326890      4 0.000552
-0.326890  -0.298531      4 0.000552
-0.298531  -0.270171      3 0.000414
-0.270171  -0.241812     10 0.001380
-0.241812  -0.213453      6 0.000828
-0.213453  -0.185093     24 0.003312
-0.185093  -0.156734     48 0.006623
-0.156734  -0.128375    101 0.013937
-0.128375  -0.100015    212 0.029253
-0.100015  -0.071656    223 0.030771
-0.071656  -0.043297    140 0.019318
-0.043297  -0.014937    188 0.025942
-0.014937   0.013422    233 0.032151
 0.013422   0.041781    256 0.035325
 0.041781   0.070141    172 0.023734
 0.070141   0.098500    145 0.020008
 0.098500   0.126859    147 0.020284
 0.126859   0.155218    117 0.016145
 0.155218   0.183578     60 0.008279
 0.183578   0.211937     30 0.004140
 0.211937   0.240296     34 0.004692
 0.240296   0.268656     34 0.004692
 0.268656   0.297015     37 0.005106
 0.297015   0.325374     72 0.009935
 0.325374   0.353734     68 0